In [3]:
# Imports
import sys
from pathlib import Path
BASE = Path.cwd().parent.parent
sys.path.insert(0, str(BASE)) # Go to base directory

from src.data import Data
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
import chromadb

In [2]:
# Get Data

d1 = Data()
d_train = d1.get_complete_data('train')

Loading ESIC DATA...
ESIC Downloaded succesfully!
Loading SQID DATA...
SQID Downloaded succesfully!
Loading Amazon Review DATA...
Amazon Review Data Downloaded succesfully!


In [4]:
BASE_DATA_DIR =  "E:/Datasets/mm-prod-search-and-reco" # Path.absolute(__file__).parent
PROD_DATA_DIR = f"{BASE_DATA_DIR}/data/prod"

client = chromadb.PersistentClient(path=f'{PROD_DATA_DIR}/chromadb')

In [5]:
collection = client.get_or_create_collection(name = "products")

In [6]:
product_data = d1.get_prod_data()

In [7]:
tfidf = TfidfVectorizer(max_features=10_000, min_df=2)

cols = [
    "product_title",
    "product_description",
    "product_bullet_point",
    "product_brand",
    "product_color"
]

prod_docs = product_data[cols].apply(lambda x: " ".join(x), axis= 1)
prod_tfidf = tfidf.fit_transform(prod_docs)

prod_tfidf_embeddings = prod_tfidf.toarray()

In [8]:
prod_docs

0        island of the blue dolphins island of  the blu...
1        the morning sidekick journal  habit tracker jo...
2        nasb giant print bible blue leathertex 2020 te...
3        learning without tears  printing power  2008 e...
4        bushnell falcon 10x50 wide angle binoculars bl...
                               ...                        
11145    vonanda faux leather sofa couch midcentury han...
11146    2021 amplified hd digital tv antenna long 180 ...
11147    65ft white christmas tree fraser fir pencil ar...
11148    womens sexy sequin dress formal gowns and even...
11149    black hoodie women oversized sweatshirt croppe...
Length: 11150, dtype: str

In [ ]:
BATCH_SIZE = 500

for i in tqdm(range(0, prod_tfidf.shape[0], BATCH_SIZE)):
    collection.add( ids = product_data["product_id"].iloc[i : i + BATCH_SIZE].to_list(),
                   embeddings= prod_tfidf_embeddings[i : i + BATCH_SIZE],
                   documents=prod_docs.iloc[i : i + BATCH_SIZE].to_list(),
                   metadatas=product_data[cols + ["image_url"]].iloc[i : i + BATCH_SIZE].to_dict("records"))
    

  0%|          | 0/23 [00:00<?, ?it/s]

In [9]:
print(collection.count())
print(collection.peek(1))

11150
{'ids': ['0547328613'], 'embeddings': array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 10000)), 'documents': ['island of the blue dolphins island of  the blue dolphins use but like new gripping tale of karana’s survival strength and courage amidst vivid descriptions of island life\nrecommended for grades 25\n192 pages hmh books for young readers blue'], 'uris': None, 'included': ['metadatas', 'documents', 'embeddings'], 'data': None, 'metadatas': [{'product_description': 'island of  the blue dolphins use but like new', 'product_brand': 'hmh books for young readers', 'image_url': 'https://m.media-amazon.com/images/I/81gSvYPU0xL._SY466_.jpg', 'product_bullet_point': 'gripping tale of karana’s survival strength and courage amidst vivid descriptions of island life\nrecommended for grades 25\n192 pages', 'product_title': 'island of the blue dolphins', 'product_color': 'blue'}]}


In [ ]:
# query_tfdif_embedding = query_train_tfidf.toarray()
# collection.query(query_embeddings=[query_tfdif_embedding[0].tolist()], n_results=10)

{'ids': [['B08L89ZGSR',
   'B07CZW7J7Z',
   'B078Y12Y6F',
   'B0777LR4ZX',
   'B07CPW4BVX',
   'B07C3C55WD',
   'B07TVJ1B2W',
   'B082YTZ4F6',
   'B01GKE7K7U',
   'B07PJL9W6D']],
 'embeddings': None,
 'documents': [['heavy duty motorcycle tool roll bag  tools not included  empty small tool bag and roll up wrench organizer tool pouch with elastic loops pocket molle webbing to wrap tool kit for mc bike atv utv ❌tools not included keep your tools safe  ready for battle · the mjölnir foldable tool roll bag from odinsberg is carefully designed by experienced motorcycle riders it’s the ideal tool kit accessory to bring on your motorbike adventure keep in your car truck or boat or just at home in your garage   · measuring 21x 95 53 x 24cm unfolded and 55 x 95 14 x 24cm folded it fits on most motorcycle tool boxes   · the 56 elastic loops have two sizes 075” and 14” to keep your wrenches sockets screwdrivers pliers and other tools measuring maximum 85” tight organized and ready for battle at a